# Stage 3 — Nakit Akışı Tahmini: SMAPE/WAPE + SHAP + Cross-Validation

Bu notebook mevcut Stage 3'ü bilimsel olarak güçlendirir:

1. **SMAPE & WAPE** — MAPE'nin sıfır patlaması sorununu çözen metrikler
2. **TimeSeriesSplit CV** — Zaman serisi verisi için doğru cross-validation
3. **SHAP Analizi** — LightGBM modelinin kararlarını açıklar
4. Karşılaştırma grafikleri

> CPU'da çalışır, GPU gerekmez.


In [ ]:
!pip install lightgbm shap scikit-learn pandas numpy matplotlib seaborn -q
print('Hazır.')

In [ ]:
# ============================================================
# 1. DRIVE BAGLANTISI — Otomatik Klasor Bulma
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

def find_colab_training_dir():
    candidates = [
        '/content/drive/MyDrive/colab_training',
        '/content/drive/My Drive/colab_training',
        '/content/drive/MyDrive/veri_seti/colab_training',
        '/content/drive/My Drive/veri_seti/colab_training',
    ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    print('Drive taraniyor...')
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        depth = root.replace('/content/drive/MyDrive', '').count(os.sep)
        if depth > 3:
            dirs[:] = []
            continue
        if 'colab_training' in dirs:
            return os.path.join(root, 'colab_training')
    return None

BASE = find_colab_training_dir()
if BASE is None:
    print('colab_training bulunamadi!')
    BASE = '/content/drive/MyDrive/colab_training'
else:
    print(f'Bulundu: {BASE}')

DATA_DIR = f'{BASE}/data'
OUT_DIR  = f'{BASE}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'DATA_DIR: {DATA_DIR}')
print(f'OUT_DIR : {OUT_DIR}')
print('Dosyalar:')
if os.path.isdir(DATA_DIR):
    for f in sorted(os.listdir(DATA_DIR)):
        sz = os.path.getsize(os.path.join(DATA_DIR, f)) / 1024**2
        print(f'  {f}  ({sz:.1f} MB)')
else:
    print('  data/ klasoru yok! Drive yapisini kontrol edin.')

In [ ]:
# ============================================================
# 2. VERİ YÜKLEME
# ============================================================
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Stage 2'den gelen zenginleştirilmiş user_monthly yükle
um_enriched = f'{OUT_DIR}/stage2_user_monthly_enriched.csv'
um_base     = f'{DATA_DIR}/stage2_user_monthly_features.csv'

if os.path.exists(um_enriched):
    user_monthly = pd.read_csv(um_enriched, encoding='utf-8-sig')
    print(f'Zenginleştirilmiş veri yüklendi: {user_monthly.shape}')
elif os.path.exists(um_base):
    user_monthly = pd.read_csv(um_base, encoding='utf-8-sig')
    print(f'Temel veri yüklendi: {user_monthly.shape}')
else:
    raise FileNotFoundError('stage2_user_monthly_features.csv bulunamadı! Önce Stage 2 notebook çalıştırın.')

print(user_monthly.columns.tolist()[:20])

In [ ]:
# ============================================================
# 3. FEATURE ENGINEERING (LAG + CYCLIC)
# ============================================================
from sklearn.preprocessing import LabelEncoder

um = user_monthly.sort_values(['user_id','year_month']).copy()

# Lag features
for lag in [1, 2, 3]:
    um[f'expense_lag{lag}'] = um.groupby('user_id')['monthly_expense_computed'].shift(lag).fillna(0)
    um[f'income_lag{lag}']  = um.groupby('user_id')['monthly_income_computed'].shift(lag).fillna(0)
    um[f'savings_lag{lag}'] = um.groupby('user_id')['savings_rate'].shift(lag).fillna(0)

# Cyclic month encoding
um['period_month'] = pd.to_datetime(um['year_month'], format='%Y-%m', errors='coerce').dt.month.fillna(1)
um['month_sin'] = np.sin(2 * np.pi * um['period_month'] / 12)
um['month_cos'] = np.cos(2 * np.pi * um['period_month'] / 12)

# Cluster encoding (varsa)
if 'cluster' in um.columns:
    um['cluster_enc'] = um['cluster'].astype(int)

# Budget style encoding (varsa)
if 'budget_style' in um.columns:
    le = LabelEncoder()
    um['budget_style_enc'] = le.fit_transform(um['budget_style'].fillna('balanced').astype(str))

print(f'Feature engineering tamamlandı: {um.shape}')

In [ ]:
# ============================================================
# 4. SUPERVISED DATASET
# ============================================================
TARGET = 'monthly_expense_computed'

# Hedef: bir sonraki ayın harcaması
um['target'] = um.groupby('user_id')[TARGET].shift(-1)
um = um.dropna(subset=['target'])

# Çok düşük harcama satırlarını filtrele (MAPE patlamasını önler)
um = um[(um[TARGET] >= 500) & (um['target'] >= 500)].copy()
print(f'Filtreden sonra: {len(um):,} satır')

# Log transform
um['target_log'] = np.log1p(um['target'])

# Feature kolonları
FEATURE_COLS = [
    'monthly_income_computed','monthly_expense_computed',
    'savings_rate','budget_utilisation','net_cashflow',
    'expense_lag1','expense_lag2','expense_lag3',
    'income_lag1','income_lag2','income_lag3',
    'savings_lag1','savings_lag2','savings_lag3',
    'month_sin','month_cos'
]
# Rolling ve spend kolonlarını ekle
roll_cols  = [c for c in um.columns if 'rolling' in c or 'peer_pct' in c or 'anomaly' in c]
spend_cols = [c for c in um.columns if c.startswith('spend_')]
extra_cols = ['cluster_enc','budget_style_enc','txn_count']
FEATURE_COLS += roll_cols + spend_cols[:8] + [c for c in extra_cols if c in um.columns]
FEATURE_COLS  = [c for c in FEATURE_COLS if c in um.columns]

X = um[FEATURE_COLS].fillna(0)
y = um['target_log']
y_orig = um['target']

print(f'X: {X.shape}  |  Özellik sayısı: {len(FEATURE_COLS)}')
print(f'y (log): min={y.min():.2f} max={y.max():.2f} mean={y.mean():.2f}')

In [ ]:
# ============================================================
# 5. METRİK FONKSİYONLARI (SMAPE, WAPE, MAPE)
# ============================================================

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — sıfıra yakında patlar!"""
    mask = y_true > 1
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

def smape(y_true, y_pred):
    """Symmetric MAPE — sıfıra sağlam, %0-200 arasında."""
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100)

def wape(y_true, y_pred):
    """Weighted Absolute Percentage Error — büyük değerlere ağırlık verir."""
    return float(np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100)

def print_all_metrics(name, y_true, y_pred):
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    m    = mape(y_true, y_pred)
    sm   = smape(y_true, y_pred)
    w    = wape(y_true, y_pred)
    print(f'  {name:<35} MAE={mae:>10,.0f}  RMSE={rmse:>10,.0f}  MAPE={m:>7.1f}%  SMAPE={sm:>7.1f}%  WAPE={w:>7.1f}%')
    return {'model': name, 'MAE': round(mae,2), 'RMSE': round(rmse,2),
            'MAPE': round(m,2), 'SMAPE': round(sm,2), 'WAPE': round(w,2)}

print('Metrik fonksiyonları hazır.')

In [ ]:
# ============================================================
# 6. TIME SERIES CROSS-VALIDATION
# ============================================================
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit

print('TimeSeriesSplit Cross-Validation başlıyor...')
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

cv_results = []
for fold, (tr_idx, te_idx) in enumerate(tscv.split(X), 1):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
    y_orig_te  = y_orig.iloc[te_idx].values
    
    model_cv = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        max_depth=8, subsample=0.8, colsample_bytree=0.7,
        random_state=42, n_jobs=-1, verbose=-1
    )
    model_cv.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
                 callbacks=[lgb.early_stopping(50, verbose=False)])
    
    preds_log = model_cv.predict(X_te)
    preds     = np.expm1(preds_log)
    
    fold_sm = smape(y_orig_te, preds)
    fold_w  = wape(y_orig_te, preds)
    cv_results.append({'fold': fold, 'smape': fold_sm, 'wape': fold_w,
                       'train_size': len(tr_idx), 'test_size': len(te_idx)})
    print(f'  Fold {fold}/{N_SPLITS}: SMAPE={fold_sm:.2f}%  WAPE={fold_w:.2f}%  (train={len(tr_idx):,}, test={len(te_idx):,})')

cv_df = pd.DataFrame(cv_results)
print(f'\nCV Ortalama SMAPE: {cv_df["smape"].mean():.2f}% ± {cv_df["smape"].std():.2f}%')
print(f'CV Ortalama WAPE : {cv_df["wape"].mean():.2f}% ± {cv_df["wape"].std():.2f}%')

In [ ]:
# ============================================================
# 7. FİNAL LightGBM MODELİ (train/test split)
# ============================================================
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te, y_orig_tr, y_orig_te = train_test_split(
    X, y, y_orig, test_size=0.15, random_state=42
)

print('Final LightGBM eğitiliyor...')
final_model = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.02, num_leaves=127,
    max_depth=10, min_child_samples=10, subsample=0.8,
    colsample_bytree=0.7, reg_alpha=0.05, reg_lambda=0.2,
    random_state=42, n_jobs=-1, verbose=-1
)
final_model.fit(
    X_tr, y_tr,
    eval_set=[(X_te, y_te)],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)]
)

preds_log  = final_model.predict(X_te)
preds      = np.expm1(preds_log)
y_te_orig  = y_orig_te.values

print('\n=== TEST METRİKLERİ ===')
metrics = print_all_metrics('LightGBM v3', y_te_orig, preds)

In [ ]:
# ============================================================
# 8. SHAP ANALİZİ
# ============================================================
import shap
import matplotlib.pyplot as plt

print('SHAP değerleri hesaplanıyor...')
explainer   = shap.TreeExplainer(final_model)

# Hız için test setinin bir örneklemini kullan
N_SHAP = min(1000, len(X_te))
X_shap = X_te.sample(N_SHAP, random_state=42)
shap_values = explainer.shap_values(X_shap)

print(f'SHAP hesaplandı: {shap_values.shape}')

# ── SHAP Summary Plot (Beeswarm) ──
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values, X_shap,
    plot_type='dot',
    max_display=20,
    show=False
)
plt.title('SHAP Beeswarm — LightGBM Nakit Akışı Tahmini\n(Her nokta bir tahmin, renk özellik değeri)', fontsize=12)
plt.tight_layout()
shap_bees_path = f'{OUT_DIR}/stage3_shap_beeswarm.png'
plt.savefig(shap_bees_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Kaydedildi: {shap_bees_path}')

In [ ]:
# ── SHAP Bar Plot (Mean Absolute) ──
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_shap,
    plot_type='bar',
    max_display=20,
    show=False
)
plt.title('SHAP Feature Importance (Ortalama |SHAP Değeri|)', fontsize=12)
plt.tight_layout()
shap_bar_path = f'{OUT_DIR}/stage3_shap_importance.png'
plt.savefig(shap_bar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Kaydedildi: {shap_bar_path}')

# SHAP feature önem skoru kaydet
shap_importance = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'mean_shap' : np.abs(shap_values).mean(0)
}).sort_values('mean_shap', ascending=False)
shap_importance.to_csv(f'{OUT_DIR}/stage3_shap_importance.csv', index=False)
print('\nTop 10 SHAP Feature:')
print(shap_importance.head(10).to_string(index=False))

In [ ]:
# ============================================================
# 9. TAHMIN vs GERÇEK GRAFİĞİ
# ============================================================
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Zaman serisi örneği (ilk 200 test noktası)
n_show = min(200, len(y_te_orig))
idx = np.arange(n_show)
axes[0].plot(idx, y_te_orig[:n_show]/1000, label='Gerçek', color='#27AE60', linewidth=1.2)
axes[0].plot(idx, preds[:n_show]/1000, label='Tahmin', color='#E74C3C', linewidth=1.0, linestyle='--')
axes[0].set_title(f'LightGBM v3 — Gerçek vs Tahmin ({n_show} örnek)', fontsize=12)
axes[0].set_xlabel('Test Örneği')
axes[0].set_ylabel('Aylık Harcama (₺ Bin)')
axes[0].legend()

# Scatter (Gerçek vs Tahmin)
max_val = max(y_te_orig.max(), preds.max())
axes[1].scatter(y_te_orig, preds, alpha=0.2, s=5, color='#3498DB')
axes[1].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Mükemmel Tahmin')
axes[1].set_title('Scatter: Gerçek vs Tahmin', fontsize=12)
axes[1].set_xlabel('Gerçek Harcama (₺)')
axes[1].set_ylabel('Tahmin (₺)')
axes[1].legend()

plt.tight_layout()
pred_path = f'{OUT_DIR}/stage3_forecast_vs_actual.png'
plt.savefig(pred_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Kaydedildi: {pred_path}')

In [ ]:
# ============================================================
# 10. CV GRAFİĞİ
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
x = cv_df['fold']
width = 0.35
ax.bar(x - width/2, cv_df['smape'], width, label='SMAPE (%)', color='#3498DB', alpha=0.85)
ax.bar(x + width/2, cv_df['wape'],  width, label='WAPE (%)',  color='#E74C3C', alpha=0.85)
ax.axhline(cv_df['smape'].mean(), color='#3498DB', linestyle='--', linewidth=1.5,
           label=f'Ort. SMAPE={cv_df["smape"].mean():.1f}%')
ax.axhline(cv_df['wape'].mean(), color='#E74C3C', linestyle='--', linewidth=1.5,
           label=f'Ort. WAPE={cv_df["wape"].mean():.1f}%')
ax.set_title('TimeSeriesSplit CV — Fold Bazlı SMAPE & WAPE', fontsize=12)
ax.set_xlabel('Fold')
ax.set_ylabel('Hata (%)')
ax.legend()
ax.set_xticks(x)
plt.tight_layout()
cv_path = f'{OUT_DIR}/stage3_cv_scores.png'
plt.savefig(cv_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Kaydedildi: {cv_path}')

In [ ]:
# ============================================================
# 11. SONUÇLARI KAYDET
# ============================================================
import pickle

pickle.dump(final_model, open(f'{OUT_DIR}/lgbm_v3_model.pkl','wb'))

stage3_results = {
    'test_metrics'  : metrics,
    'cv_results'    : cv_results,
    'cv_smape_mean' : float(cv_df['smape'].mean()),
    'cv_smape_std'  : float(cv_df['smape'].std()),
    'cv_wape_mean'  : float(cv_df['wape'].mean()),
    'cv_wape_std'   : float(cv_df['wape'].std()),
    'n_features'    : len(FEATURE_COLS),
    'feature_cols'  : FEATURE_COLS,
    'shap_top10'    : shap_importance.head(10).to_dict('records')
}

with open(f'{OUT_DIR}/stage3_metrics.json','w',encoding='utf-8') as f:
    json.dump(stage3_results, f, indent=2, ensure_ascii=False)

print('\n=== STAGE 3 ÖZET ===')
print(f'  Test SMAPE      : {metrics["SMAPE"]:.2f}%  (MAPE yerine)')
print(f'  Test WAPE       : {metrics["WAPE"]:.2f}%')
print(f'  Test MAE        : ₺{metrics["MAE"]:,.0f}')
print(f'  CV SMAPE        : {cv_df["smape"].mean():.2f}% ± {cv_df["smape"].std():.2f}%')
print(f'  SHAP Açıklaması : Evet (Top özellik: {shap_importance.iloc[0]["feature"]})')
print(f'\nBu dosyaları indirip projenizin reports/ ve models/ klasörüne aktarın.')